# Data loading

In [1]:
import numpy as np
import os

# TODO: Jakub - add loading data from Google disk
# Following function generates dummy dataset with materials descriptors
# In future it will be replaced by code for loading datasets from Google drive and processing them
def gen_dummy_data(sample_num, output_dir, porosity):
    modulus = np.random.rand(sample_num) * 70
    with open(os.path.join(output_dir, "z_modulus"), "w") as f:
        for i in range(sample_num):
            f.write(str(modulus[i]))
            if not i == sample_num - 1:
                f.write("\n")
    for i in range(sample_num):
        data = np.random.choice(a=[False, True], size=(10, 10, 10), p=[porosity, 1-porosity])
        np.save(os.path.join(output_dir, "data_{}".format(i)), data)

gen_dummy_data(30, "dummy", 0.5)

# Compute descriptors

## Prepare function for calculating filtrations and topo descriptors

In [2]:
import numpy as np
import subprocess
import gudhi as gd
from gudhi.representations import DiagramSelector, PersistenceImage

# Function for calculating 
def get_cone_filtration(input_path, output_path):
    # Python sucks. Julia rules!
    # Calling julia functions from python is problematic, so we just run script instead
    result = subprocess.run(['julia', '../filtrations/julia/cone_filtrator.jl', input_path, output_path])

# Function for calculating vectorised PH based on filtered cubical complex
def get_vectorized_pers_from_filtration(filtration_path, output_path, resolution=(10,10)):
    filtration = np.load(filtration_path)
    comp = gd.PeriodicCubicalComplex(top_dimensional_cells=filtration, periodic_dimensions=[False, False, True])
    comp.compute_persistence()
    persist_imager = PersistenceImage(resolution=resolution)
    gudhi_diag_selector = gd.representations.DiagramSelector(use=True,limit=np.inf, point_type="finite")
    output = []
    for i in range(3):
        intervals = gudhi_diag_selector(comp.persistence_intervals_in_dimension(i))
        if len(intervals) == 0:
             output.append(np.zeros(resolution[0] * resolution[0]))
             continue
        img_pers = persist_imager(intervals)
        output.append(img_pers)
    np.save(output_path, output)


## Compute filtrations and topo descriptors

In [3]:
import sys
sys.path.append("..")
from pipelines import DescriptorPipeline

"""
Config file that contains
1. Location of datasets (config files)
2. Number of workers
3. Tasks to perform
""" 
DESCRIPTOR_CONFIG_PATH = "desc.json"
pipeline_descriptor = DescriptorPipeline(DESCRIPTOR_CONFIG_PATH, get_cone_filtration,
                                         get_vectorized_pers_from_filtration)
pipeline_descriptor.run()

Dataset dummy processed


# Train and evaluate models

In [4]:
from sklearn.ensemble import RandomForestRegressor
from pipelines import AnalysisPipeline
# Set Gride Search parameters json path
MODEL_CONFIG_PATH = "model.json"
"""
Config file that contains
1. Location of data (descriptors, volume fraction + labels)
2. Number of workers
3. Test set size
4. Output file name
""" 
DATA_CONFIG_PATH = "analisis.json"
pipeline_analysis = AnalysisPipeline(MODEL_CONFIG_PATH, DATA_CONFIG_PATH, RandomForestRegressor())
# Start training and evaluation. Results are saved in file
pipeline_analysis.run()